In [101]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path

In [102]:
# glac_df = pd.read_parquet("/wolke_scratch/dnikolo/ICON_Data/sedim_search/R2B4/wbf_v3/postprocessing_output/np/20070301.0000_20070315.0000/Agg_03_T_18_12.parquet")
glac_df = pd.read_parquet("/wolke_scratch/dnikolo/ICON_Data/sedim_search/R2B4/wbf_v3/postprocessing_output/Final_results/2007_glac_04.parquet")

In [103]:
bulk_cloud_data = xr.open_dataset("/wolke_scratch/dnikolo/ICON_Data/sedim_search/R2B4/wbf_v3/bulk_data/sedim_r2b4_wbfv3_bulk_cloud_ml_20070101T000000Z.nc")

In [104]:
test_cloud = glac_df.iloc[10]

In [105]:
test_cloud.track_length

Timedelta('0 days 04:00:00')

In [113]:


BULK_DATA_DIR = Path(
    "/wolke_scratch/dnikolo/ICON_Data/sedim_search/"
    "R2B4/wbf_v3/bulk_data"
)

def nanmean_or_nan(values, axis=None):
    values = np.asarray(values, dtype=float)

    valid_count = np.sum(~np.isnan(values), axis=axis)
    value_sum = np.nansum(values, axis=axis)

    result = np.full(
        np.shape(value_sum),
        np.nan,
        dtype=float,
    )

    return np.divide(
        value_sum,
        valid_count,
        out=result,
        where=valid_count > 0,
    )

def timestamp_to_filename(timestamp):
    timestamp = pd.Timestamp(timestamp)

    filename = (
        "sedim_r2b4_wbfv3_bulk_cloud_ml_"
        f"{timestamp:%Y%m%dT000000}Z.nc"
    )

    return BULK_DATA_DIR / filename

def timestamp_to_model_time(timestamp):
    timestamp = pd.Timestamp(timestamp)
    fraction = (timestamp - timestamp.normalize()) / pd.Timedelta(days=1)

    return round(
        int(timestamp.strftime("%Y%m%d")) + fraction,
        6,  # coordinate is stored to six decimal places
    )
def extract_bulk_values(row):
    filename = None
    if pd.isna(row.tracknumber):
        row["bulk_values"]=None
        return 
    bulk_values = np.full((int(row.track_length/pd.Timedelta(minutes=15))+1,25),np.nan)
    bulk_data_ds=None
    for timestep, ncell_idxs in enumerate(row.model_grid_ncell):
        timestamp = row.track_start_time + pd.Timedelta(minutes=15)*timestep
        new_filename = timestamp_to_filename(timestamp)
        
        if new_filename != current_filename:
                if bulk_data_ds is not None:
                    bulk_data_ds.close()
                bulk_data_ds = xr.open_dataset(filename)
        bulk_values[timestep,:] = nanmean_or_nan(bulk_data_ds.ze.sel(ncells=ncell_idxs,time=timestamp_to_model_time(timestamp),method='nearest').values,axis=1)
    row["bulk_values"] =bulk_values

In [ ]:
dataset_cache = {}


def get_bulk_dataset(filename):
    if filename not in dataset_cache:
        dataset_cache[filename] = xr.open_dataset(filename)

    return dataset_cache[filename]


def extract_bulk_values_cached(row):
    if pd.isna(row.tracknumber):
        row["bulk_values"] = None
        return row

    n_timesteps = len(row.model_grid_ncell)
    bulk_values = np.full((n_timesteps, 25), np.nan)

    for timestep, ncell_idxs in enumerate(row.model_grid_ncell):
        # print(f"Analyzing {timestep}/{bulk_values.shape[0]} - {ncell_idxs.shape}")
        timestamp = (
            pd.Timestamp(row.track_start_time)
            + pd.Timedelta(minutes=15 * timestep)
        )

        filename = timestamp_to_filename(timestamp)
        bulk_data_ds = get_bulk_dataset(filename)

        selected = bulk_data_ds.ze.sel(
            ncells=ncell_idxs,
            time=timestamp_to_model_time(timestamp),
            method="nearest",
        )

        bulk_values[timestep] = nanmean_or_nan(
            selected.values,
            axis=1,
        )

    row["bulk_values"] = bulk_values
    return row

In [117]:
try:
    glac_df = glac_df.progress_apply(
        extract_bulk_values_cached,
        axis=1,
    )
finally:
    for dataset in dataset_cache.values():
        dataset.close()

    dataset_cache.clear()

Extracting bulk values:   0%|          | 0/297797 [00:00<?, ?it/s]

Analyzing 0/30 - (1,)
Analyzing 1/30 - (1,)
Analyzing 2/30 - (1,)
Analyzing 3/30 - (2,)
Analyzing 4/30 - (2,)
Analyzing 5/30 - (1,)
Analyzing 6/30 - (1,)
Analyzing 7/30 - (1,)
Analyzing 8/30 - (1,)
Analyzing 9/30 - (1,)
Analyzing 10/30 - (1,)
Analyzing 11/30 - (1,)
Analyzing 12/30 - (1,)
Analyzing 13/30 - (1,)
Analyzing 14/30 - (1,)
Analyzing 15/30 - (1,)
Analyzing 16/30 - (1,)
Analyzing 17/30 - (1,)
Analyzing 18/30 - (1,)
Analyzing 19/30 - (1,)
Analyzing 20/30 - (1,)
Analyzing 21/30 - (1,)
Analyzing 22/30 - (1,)
Analyzing 23/30 - (1,)
Analyzing 24/30 - (1,)
Analyzing 25/30 - (1,)
Analyzing 26/30 - (1,)
Analyzing 27/30 - (1,)
Analyzing 28/30 - (1,)
Analyzing 29/30 - (1,)
Analyzing 0/8 - (1,)
Analyzing 1/8 - (1,)
Analyzing 2/8 - (1,)
Analyzing 3/8 - (2,)
Analyzing 4/8 - (2,)
Analyzing 5/8 - (2,)
Analyzing 6/8 - (1,)
Analyzing 7/8 - (1,)
Analyzing 0/18 - (2,)
Analyzing 1/18 - (3,)
Analyzing 2/18 - (3,)
Analyzing 3/18 - (1,)
Analyzing 4/18 - (1,)
Analyzing 5/18 - (1,)
Analyzing 6/18 - (1,

KeyboardInterrupt: 

In [ ]:
from tqdm.auto import tqdm

tqdm.pandas(desc="Extracting bulk values")

glac_df = glac_df.progress_apply(
    extract_bulk_values,
    axis=1,
)

Extracting bulk values:   0%|          | 0/8770 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
glac_df.apply(extract_bulk_values,axis=1)

KeyboardInterrupt: 